# Sportmonks Datenakquise — Matchstatistiken Super League

Ergänzt die API-Football Basisdaten um **detaillierte Matchstatistiken** (Sportmonks Football API v3).

**Zusätzliche Daten pro Spiel & Team:**  
🔵 Ballbesitz · 🎯 Schüsse (gesamt/aufs Tor/daneben/geblockt) · 🟨 Karten  
🚩 Eckbälle · 💢 Fouls · ↗️ Abseits · 🧤 Saves · ⚡ Angriffe · 🔁 Pässe

**Voraussetzung:** `.env` im Projekt-Root:
```
API_SPORTMONKS_KEY="dein_key"
API_SPORTMONKS_URL="https://api.sportmonks.com/v3/football"
```

**Output-Dateien:**
- `raw/fixture_statistics.csv` — eine Zeile pro Team pro Spiel
- `raw/team_stats_sportmonks.csv` — Saisondurchschnitte pro Team
- `raw/season_timeline.csv` — kumulierte Punkte pro Spieltag

---
**Score-Struktur (v3):** Jeder Eintrag im `scores`-Array hat `description` (`1ST_HALF`, `2ND_HALF`, `CURRENT`) und `score.participant` (`home`/`away`).  
`CURRENT` = Endstand bei abgeschlossenen Spielen.  
**State-Struktur (v3):** `include=state` liefert Objekt mit `short_name` (`FT`, `NS`, `AET`, …).  
**Statistiken:** `type_id`-basiert, nur tatsächlich erfasste Werte, via `location: home|away`.

## 1. Setup & Imports

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("__file__").resolve().parent.parent / ".env"
load_dotenv(dotenv_path=env_path)

API_KEY  = os.environ["API_SPORTMONKS_KEY"]
BASE_URL = os.environ.get("API_SPORTMONKS_URL", "https://api.sportmonks.com/v3/football").rstrip("/")

if "docs.sportmonks" in BASE_URL:
    raise ValueError(
        "API_SPORTMONKS_URL zeigt auf die Doku, nicht die API!\n"
        "Korrekt: API_SPORTMONKS_URL=https://api.sportmonks.com/v3/football"
    )

HEADERS = {"Authorization": API_KEY}
RAW_DIR = Path("raw")
RAW_DIR.mkdir(exist_ok=True)

print(f"Base URL : {BASE_URL}")
print(f"API Key  : {'✓ geladen' if API_KEY else '✗ FEHLT'}")
print(f"Output   : {RAW_DIR.resolve()}")

## 2. Hilfsfunktionen

In [ ]:
def api_get(endpoint: str, params: dict = None) -> dict:
    """Einzelner GET-Request an Sportmonks v3. Gibt das vollständige JSON zurück."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    resp = requests.get(url, headers=HEADERS, params=params or {})
    resp.raise_for_status()
    return resp.json()


def api_get_all(endpoint: str, params: dict = None) -> list:
    """Paginierter GET: lädt alle Seiten (via has_more) und gibt flache Liste zurück."""
    params = {**(params or {}), "per_page": 50}
    all_data, page = [], 1

    while True:
        data = api_get(endpoint, {**params, "page": page})
        items = data.get("data", [])
        all_data.extend(items)

        # Rate-limit aus Meta anzeigen
        meta = data.get("subscription", [{}])
        rate_info = data.get("rate_limit", {})
        remaining = rate_info.get("remaining", "?")
        pagination = data.get("pagination", {})
        has_more   = pagination.get("has_more", False)

        print(f"  Seite {page:>2} | +{len(items):>3} Einträge | Rate verbleibend: {remaining}")

        if not has_more:
            break
        page += 1
        time.sleep(0.4)

    return all_data


# ---------------------------------------------------------------------------
# Parsing-Hilfsfunktionen (korrigiert für v3-Datenstruktur)
# ---------------------------------------------------------------------------

def parse_participants(participants: list) -> tuple[dict, dict]:
    """Gibt (home_team, away_team) zurück. Location steht in participant.meta.location."""
    home = {"team_id": None, "team_name": ""}
    away = {"team_id": None, "team_name": ""}
    for p in participants or []:
        loc = (p.get("meta") or {}).get("location", "")
        entry = {"team_id": p["id"], "team_name": p.get("name", "")}
        if loc == "home":
            home = entry
        elif loc == "away":
            away = entry
    return home, away


def parse_scores(scores: list) -> dict:
    """Extrahiert Heim- und Auswärtstore für alle Abschnitte.

    v3-Struktur: Jeder Eintrag hat description (1ST_HALF, 2ND_HALF, CURRENT)
    und score.participant (home | away). CURRENT = Endstand bei FT-Spielen.
    Zwei Einträge pro description – einer für home, einer für away.
    """
    result = {}
    target_descriptions = {"CURRENT", "1ST_HALF", "2ND_HALF"}

    for sc in scores or []:
        desc        = sc.get("description", "").upper()
        score_obj   = sc.get("score", {})
        participant = score_obj.get("participant", "")  # "home" | "away"
        goals       = score_obj.get("goals")            # int

        if desc not in target_descriptions or participant not in ("home", "away"):
            continue

        label = f"score_{desc.lower()}_{participant}"  # z.B. score_current_home
        result[label] = goals

    # Aliase für Endstand (CURRENT = Vollzeit bei FT)
    result["goals_home"] = result.get("score_current_home")
    result["goals_away"] = result.get("score_current_away")
    return result


def parse_state(state_field) -> str:
    """Gibt short_name des States zurück (z.B. 'FT', 'NS', 'AET').

    v3: include=state liefert ein Objekt {id, state, name, short_name, developer_name}.
    Fallback auf leeren String wenn State nicht included oder None.
    """
    if isinstance(state_field, dict):
        return state_field.get("short_name", "") or state_field.get("developer_name", "")
    return ""


def parse_statistics(statistics: list, type_map: dict) -> tuple[dict, dict]:
    """Pivotiert die type_id-basierte Statistikliste in (home_stats, away_stats).

    v3: Nur tatsächlich erfasste Werte werden zurückgegeben (keine Nullen).
    location: 'home' | 'away'
    """
    home_stats: dict = {}
    away_stats: dict = {}
    for stat in statistics or []:
        type_id  = stat.get("type_id")
        location = stat.get("location", "")
        value    = (stat.get("data") or {}).get("value")
        col_name = type_map.get(type_id, f"stat_type_{type_id}")
        if location == "home":
            home_stats[col_name] = value
        elif location == "away":
            away_stats[col_name] = value
    return home_stats, away_stats

print("✓ Hilfsfunktionen bereit")

## 3. Statistik-Typen laden (type_id → Spaltenname)

In [ ]:
def make_col_name(name: str) -> str:
    """Konvertiert einen Statistik-Namen in einen sauberen Python-Spaltennamen."""
    return (
        name.lower()
        .replace(" ", "_").replace("/", "_").replace("-", "_")
        .replace("(", "").replace(")", "").replace("%", "pct")
        .replace(".", "").replace("'", "")
    )


print("Lade Statistik-Typen via /types ...")
# Versuche zuerst entity-spezifisch, dann alle Typen als Fallback
try:
    types_raw = api_get_all("types/entity/fixture")
    print(f"  → /types/entity/fixture: {len(types_raw)} Typen")
except Exception:
    print("  → /types/entity/fixture nicht verfügbar, lade alle Typen...")
    types_raw = api_get_all("types")
    print(f"  → /types: {len(types_raw)} Typen total")

TYPE_MAP: dict[int, str] = {}
for t in types_raw:
    tid  = t["id"]
    name = t.get("name") or t.get("developer_name") or f"type_{tid}"
    TYPE_MAP[tid] = make_col_name(name)

print(f"\n{len(TYPE_MAP)} Typen im Mapping.")
print("\nAlle geladenen Statistik-Typen (ID → Spaltenname):")
for tid, col in sorted(TYPE_MAP.items()):
    print(f"  {tid:>5}: {col}")

## 4. Swiss Super League finden

In [ ]:
print("Lade Schweizer Ligen (/leagues mit countryCode:CH Filter)...")
leagues_raw = api_get_all(
    "leagues",
    params={"filters": "countryCode:CH", "include": "currentSeason"}
)

rows = []
for lg in leagues_raw:
    current = lg.get("currentSeason") or {}
    rows.append({
        "league_id":   lg["id"],
        "name":        lg.get("name", ""),
        "short_code":  lg.get("short_code", ""),
        "type":        lg.get("type", ""),
        "season_id":   current.get("id"),
        "season_name": current.get("name"),
    })

df_leagues = pd.DataFrame(rows)
print("\nSchweizer Ligen:")
print(df_leagues[["league_id", "name", "type", "season_id", "season_name"]].to_string(index=False))

In [ ]:
# Super League automatisch erkennen (Fallback: manuell setzen)
sl_row = df_leagues[df_leagues["name"].str.contains("Super League", case=False, na=False)]

if sl_row.empty:
    # ── MANUELL SETZEN falls automatische Erkennung fehlschlägt ──
    LEAGUE_ID = None   # z.B. 1034
    SEASON_ID = None   # z.B. 23614
    print("❌ 'Super League' nicht gefunden — bitte LEAGUE_ID und SEASON_ID manuell eintragen!")
    print("   Tabelle oben prüfen und Werte einsetzen.")
else:
    LEAGUE_ID = int(sl_row.iloc[0]["league_id"])
    SEASON_ID = int(sl_row.iloc[0]["season_id"])
    print(f"✅ Swiss Super League gefunden!")
    print(f"   League ID : {LEAGUE_ID}")
    print(f"   Season ID : {SEASON_ID}  ({sl_row.iloc[0]['season_name']})")

## 5. Fixtures mit Statistiken laden

In [ ]:
assert LEAGUE_ID and SEASON_ID, "LEAGUE_ID / SEASON_ID nicht gesetzt — Zelle 4 prüfen!"

print(f"Lade alle Fixtures für Season {SEASON_ID} mit Statistiken...\n")
print("Includes: statistics · participants · scores · state")
print("(Das kann mehrere Seiten dauern — bitte warten)\n")

fixtures_raw = api_get_all(
    f"fixtures/seasons/{SEASON_ID}",
    params={"include": "statistics;participants;scores;state"}
)

n_total = len(fixtures_raw)
n_ft    = sum(1 for fx in fixtures_raw if parse_state(fx.get("state")) in ("FT", "AET", "PEN"))
print(f"\n{n_total} Fixtures geladen ({n_ft} abgeschlossen, {n_total - n_ft} ausstehend).")

## 6. Fixtures flachklopfen & Statistiken pivotieren

In [ ]:
FINISHED_STATES = {"FT", "AET", "PEN", "BREAK", "ET"}

home_rows, away_rows, skipped = [], [], 0

for fx in fixtures_raw:
    status = parse_state(fx.get("state"))
    if status not in FINISHED_STATES:
        skipped += 1
        continue

    home_team, away_team = parse_participants(fx.get("participants", []))
    score_data           = parse_scores(fx.get("scores", []))
    home_stats, away_stats = parse_statistics(fx.get("statistics", []), TYPE_MAP)

    base = {
        "fixture_id":    fx["id"],
        "date":          fx.get("starting_at", ""),
        "round_id":      fx.get("round_id"),
        "status":        status,
        "home_team_id":  home_team["team_id"],
        "home_team":     home_team["team_name"],
        "away_team_id":  away_team["team_id"],
        "away_team":     away_team["team_name"],
        # Scores (alle Abschnitte)
        **score_data,
    }

    # Home-Zeile: Perspektive des Heimteams
    home_row = {
        **base,
        "perspective":    "home",
        "team_id":        home_team["team_id"],
        "team_name":      home_team["team_name"],
        "goals_scored":   score_data.get("goals_home"),
        "goals_conceded": score_data.get("goals_away"),
        **home_stats,
    }
    home_rows.append(home_row)

    # Away-Zeile: Perspektive des Auswärtsteams
    away_row = {
        **base,
        "perspective":    "away",
        "team_id":        away_team["team_id"],
        "team_name":      away_team["team_name"],
        "goals_scored":   score_data.get("goals_away"),
        "goals_conceded": score_data.get("goals_home"),
        **away_stats,
    }
    away_rows.append(away_row)

df_fixture_stats = (
    pd.concat([pd.DataFrame(home_rows), pd.DataFrame(away_rows)], ignore_index=True)
    .sort_values(["fixture_id", "perspective"])
    .reset_index(drop=True)
)

df_fixture_stats.to_csv(RAW_DIR / "fixture_statistics.csv", index=False)

n_fixtures = len(df_fixture_stats) // 2
stat_cols  = [c for c in df_fixture_stats.columns if c not in {
    "fixture_id", "date", "round_id", "status",
    "home_team_id", "home_team", "away_team_id", "away_team",
    "perspective", "team_id", "team_name",
    "goals_home", "goals_away", "goals_scored", "goals_conceded",
    "score_current_home", "score_current_away",
    "score_1st_half_home", "score_1st_half_away",
    "score_2nd_half_home", "score_2nd_half_away",
}]

print(f"✅ fixture_statistics.csv gespeichert")
print(f"   {n_fixtures} Spiele | {skipped} ausstehend/übersprungen")
print(f"   {len(df_fixture_stats.columns)} Spalten total, davon {len(stat_cols)} Statistik-Spalten")
print(f"\nStatistik-Spalten verfügbar:")
print("  " + ", ".join(stat_cols))

df_fixture_stats.head(4)

## 7. Team-Aggregat: Saisondurchschnitte

Grundlage für **Radar Chart** und **Scatter Plot** (Ballbesitz vs. Punkte).

In [ ]:
exclude_from_agg = {
    "fixture_id", "date", "round_id", "status",
    "home_team_id", "home_team", "away_team_id", "away_team",
    "perspective", "team_id", "team_name",
    "score_current_home", "score_current_away",
    "score_1st_half_home", "score_1st_half_away",
    "score_2nd_half_home", "score_2nd_half_away",
}

numeric_cols = [
    c for c in df_fixture_stats.columns
    if c not in exclude_from_agg
    and c not in {"team_id", "team_name"}
    and pd.api.types.is_numeric_dtype(df_fixture_stats[c])
]

agg_spec = {col: "mean" for col in numeric_cols}
agg_spec["fixture_id"] = "count"

df_team_agg = (
    df_fixture_stats
    .groupby(["team_id", "team_name"])
    .agg(agg_spec)
    .rename(columns={"fixture_id": "games_played", **{c: f"{c}_avg" for c in numeric_cols}})
    .reset_index()
)

df_team_agg.to_csv(RAW_DIR / "team_stats_sportmonks.csv", index=False)

print(f"✅ team_stats_sportmonks.csv gespeichert")
print(f"   {len(df_team_agg)} Teams | {len(df_team_agg.columns)} Spalten")

# Vorschau: Ballbesitz, Schüsse, Angriffe
preview = ["team_name", "games_played"] + [
    c for c in df_team_agg.columns
    if any(kw in c for kw in ["possession", "shot", "corner", "attack", "foul", "save"])
][:12]
print()
df_team_agg[preview].sort_values("games_played", ascending=False)

## 8. Saisonverlauf — kumulierte Punkte pro Team

Grundlage für das **Liniendiagramm** im Blog.

In [ ]:
def match_points(goals_scored, goals_conceded) -> int | None:
    """3 = Sieg, 1 = Unentschieden, 0 = Niederlage."""
    if pd.isna(goals_scored) or pd.isna(goals_conceded):
        return None
    gs, gc = int(goals_scored), int(goals_conceded)
    return 3 if gs > gc else (1 if gs == gc else 0)


timeline_rows = []
for _, row in df_fixture_stats.iterrows():
    pts = match_points(row.get("goals_scored"), row.get("goals_conceded"))
    timeline_rows.append({
        "fixture_id":    row["fixture_id"],
        "date":          row["date"],
        "round_id":      row["round_id"],
        "team_id":       row["team_id"],
        "team_name":     row["team_name"],
        "perspective":   row["perspective"],
        "goals_scored":  row.get("goals_scored"),
        "goals_conceded": row.get("goals_conceded"),
        "points":        pts,
        "win":           1 if pts == 3 else 0,
        "draw":          1 if pts == 1 else 0,
        "loss":          1 if pts == 0 else 0,
    })

df_timeline = pd.DataFrame(timeline_rows)
df_timeline["date"] = pd.to_datetime(df_timeline["date"], utc=True)
df_timeline = df_timeline.sort_values(["team_name", "date"]).reset_index(drop=True)

# Kumulierte Werte pro Team
for col in ("points", "goals_scored", "goals_conceded", "win", "draw", "loss"):
    df_timeline[f"{col}_cumulative"] = df_timeline.groupby("team_id")[col].cumsum()

df_timeline["match_nr"] = df_timeline.groupby("team_id").cumcount() + 1

df_timeline.to_csv(RAW_DIR / "season_timeline.csv", index=False)

print(f"✅ season_timeline.csv gespeichert")
print(f"   {len(df_timeline)} Zeilen (1 pro Team pro Spiel)")

# Vorschau FC Thun
thun = df_timeline[df_timeline["team_name"].str.contains("Thun", case=False, na=False)]
if not thun.empty:
    print(f"\nFC Thun — letzten 5 Spiele:")
    cols = ["match_nr", "date", "perspective", "goals_scored", "goals_conceded",
            "points", "points_cumulative"]
    print(thun[cols].tail(5).to_string(index=False))
else:
    print("\nℹ️  'Thun' nicht gefunden — Teamnamen aus der Fixtures-Tabelle prüfen.")
    print("   Verfügbare Teams:", df_timeline["team_name"].unique().tolist())

## 9. Kombinierter Datensatz: Sportmonks + API-Football

Merged Saisondurchschnitte aus Sportmonks mit Tabellendaten aus API-Football  
→ Grundlage für den **Scatter Plot: Ballbesitz vs. Punkte**.

In [ ]:
standings_path = RAW_DIR / "standings.csv"

if not standings_path.exists():
    print("ℹ️  standings.csv nicht gefunden.")
    print("   Zuerst fetch_data.ipynb (API-Football) ausführen, dann diese Zelle neu starten.")
else:
    df_standings = pd.read_csv(standings_path)

    # Join über Teamnamen (Fallback: manuelles Mapping nötig falls Namen abweichen)
    df_combined = df_standings.merge(
        df_team_agg,
        on="team_name",
        how="left",
        suffixes=("_apifootball", "_sportmonks")
    )

    df_combined.to_csv(RAW_DIR / "teams_combined.csv", index=False)

    print(f"✅ teams_combined.csv gespeichert")
    print(f"   {len(df_combined)} Teams | {len(df_combined.columns)} Spalten")
    print(f"   Nicht gematchte Teams (NaN in Sportmonks-Spalten):")
    unmatched = df_combined[df_combined["games_played"].isna()]["team_name"].tolist()
    if unmatched:
        print(f"   {unmatched}")
        print("   → Teamnamen zwischen den APIs unterscheiden sich — manuelles Mapping prüfen.")
    else:
        print("   Alle Teams erfolgreich gematchet! ✓")

    # Vorschau Scatter-Plot-Daten
    scatter_cols = ["rank", "team_name", "points"] + [
        c for c in df_combined.columns if "possession" in c or "ball" in c
    ]
    if len(scatter_cols) > 3:
        print()
        print(df_combined[scatter_cols].sort_values("rank").to_string(index=False))

## 10. Übersicht aller gespeicherten Dateien

In [ ]:
print("Gespeicherte Dateien in data_acquisition/raw/:\n")
for f in sorted(RAW_DIR.glob("*.csv")):
    df   = pd.read_csv(f)
    size = f.stat().st_size / 1024
    print(f"  {f.name:<38} {len(df):>4} Zeilen × {len(df.columns):>3} Spalten   ({size:.1f} KB)")

print("\n🏁 Sportmonks Datenakquise abgeschlossen.")
print("   Nächster Schritt: uv run python eda/generate-data-profile.py")